In [1]:
import pandas as pd
import numpy as np
import os

print("Starting DLS & Wicket Impact Analysis...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# ---------------------------------------------------------
# PART A: Rain / DLS Flags
# ---------------------------------------------------------
# The 'method' column usually contains 'D/L' for rain-affected matches
if 'method' in matches.columns:
    matches['is_dls'] = matches['method'].apply(lambda x: 1 if pd.notna(x) and 'D/L' in str(x) else 0)
else:
    matches['is_dls'] = 0
    print("Note: 'method' column not found, setting DLS flag to 0.")

dls_flags_df = matches[['id', 'is_dls']].rename(columns={'id': 'match_id'})

# ---------------------------------------------------------
# PART B: Wicket Impact Analysis (2nd Innings / Chases)
# ---------------------------------------------------------
chase_df = deliveries[deliveries['inning'] == 2].copy()

# Identify if the chase was successful
chase_df = chase_df.merge(matches[['id', 'winner']], left_on='match_id', right_on='id')
chase_df['chase_successful'] = (chase_df['batting_team'] == chase_df['winner']).astype(int)

# Check if a wicket fell on any given ball
if 'is_wicket' in chase_df.columns:
    chase_df['wicket_event'] = chase_df['is_wicket']
else:
    chase_df['wicket_event'] = chase_df['player_dismissed'].notna().astype(int)

# Group by match and over to see cumulative wickets down
over_state = chase_df.groupby(['match_id', 'over']).agg(
    wickets_in_over=('wicket_event', 'sum'),
    chase_successful=('chase_successful', 'max')
).reset_index()

over_state['total_wickets_down'] = over_state.groupby('match_id')['wickets_in_over'].cumsum()

# Define Match Phases
def get_over_phase(over):
    if over <= 5: return 'Powerplay (0-5)'
    elif over <= 14: return 'Middle (6-14)'
    else: return 'Death (15-19)'

over_state['phase'] = over_state['over'].apply(get_over_phase)

# Calculate historical win probability for every wicket state in every phase
wicket_impact_matrix = over_state.groupby(['phase', 'total_wickets_down']).agg(
    situations=('match_id', 'count'),
    wins=('chase_successful', 'sum')
).reset_index()

# Filter out extremely rare situations (like being 9 down in the powerplay) to avoid noisy data
wicket_impact_matrix = wicket_impact_matrix[wicket_impact_matrix['situations'] >= 20]
wicket_impact_matrix['win_prob_pct'] = (wicket_impact_matrix['wins'] / wicket_impact_matrix['situations'] * 100).round(1)

# Calculate the actual "Impact" (How much win % drops if you lose one more wicket)
# We shift the column by -1 to subtract the win % of the NEXT wicket state
wicket_impact_matrix['prob_drop_if_next_wicket_falls'] = wicket_impact_matrix.groupby('phase')['win_prob_pct'].diff(-1) * -1

# ---------------------------------------------------------
# PART C: Save the Data
# ---------------------------------------------------------
os.makedirs('../data/processed', exist_ok=True)
dls_flags_df.to_csv('../data/processed/dls_flags.csv', index=False)
wicket_impact_matrix.to_csv('../data/processed/wicket_impact.csv', index=False)

print("✅ DLS Flags and Wicket Impact calculated successfully!")
print("\nSample Wicket Impact in Middle Overs:")
print("(Look at 'prob_drop_if_next_wicket_falls' - this is exactly what we will show in the UI Alert!)")
display(wicket_impact_matrix[wicket_impact_matrix['phase'] == 'Middle (6-14)'].head(6))

Starting DLS & Wicket Impact Analysis...
✅ DLS Flags and Wicket Impact calculated successfully!

Sample Wicket Impact in Middle Overs:
(Look at 'prob_drop_if_next_wicket_falls' - this is exactly what we will show in the UI Alert!)


,phase,total_wickets_down,situations,wins,win_prob_pct,prob_drop_if_next_wicket_falls
11,Middle (6-14),0,641,523,81.6,-3.5
12,Middle (6-14),1,1741,1360,78.1,-13.5
13,Middle (6-14),2,2349,1518,64.6,-15.1
14,Middle (6-14),3,2149,1064,49.5,-16.0
15,Middle (6-14),4,1371,459,33.5,-13.7
16,Middle (6-14),5,753,149,19.8,-9.2
